# Stata Programming Webinar

From typing commands to reusable Stata programs

June 2026 Public Webinar

## What we will discuss

- Think about a practical project layout for Stata work
- A backup plan that combines backup and version history
- A mental model for `do`, `ado`, Mata, and plugins
- Serveral programming tips, handling file path, passing inputs, returning results, and Mata matrix subscripting

## Audience

This webinar is for Stata users who can run commands and do-files, and now want their analysis projects to be easier to rerun, review, extend, and share.

## Plan the project

1. Protect the project: data privacy, code privacy, backup, and sharing file 
2. Choose the right programming surface: do, ado, mata, and plugin
3. Deal with folder structures of the project, input and output files, absolute and relative path  
4. Move values through code deliberately: local macro, global macro, scalar, pass arguments, return results
5. Design output

## Sample project layout

```text
data/   # small public sample datasets
src/    # Stata do-files, ado files, and helper scripts
docs/   # slides, notes, and supporting documentation
```

The layout should make the first rerun boring in the best possible way. It should also sustain the moving of the project to a different machine and to a collaborator. 

## 1. Protect the work

Every project needs two safety nets:

- Backup recovers a lost or deleted file
- Version control explains what changed, when, and why
- GitHub adds review, collaboration, issues, and release history

Use both. They solve different problems. Data privacy requirement will complicates the situation.

## Backup Is not revision control

| Question | Backup | Git/GitHub |
| --- | --- | --- |
| Can I meet data privacy requirement? | Maybe | Maybe |
| Can I recover the file? | Yes | Usually |
| Can I compare two versions? | Sometimes | Yes |
| Can I branch an experiment? | No | Yes |
| Can collaborators review changes? | No | Yes |
| Can I explain the project history? | No | Yes |

## 2. Programming tools

A good Stata setup makes structure visible:

- Use an editor with Stata syntax highlighting
- Keep code, data, and documents in predictable folders
- Within project's data privacy compliance, use AI coding agents for scaffolding, review, and refactoring
- Treat generated code like a student's draft: inspect and test it

## 3. Choose the right Stata surface

| Surface | Best Use | Effort |
| --- | --- | --- |
| `.do` | Reproducible analysis and reporting scripts | Low |
| `.ado` | Reusable commands that behave like Stata commands | Medium |
| Mata | Matrix-heavy or performance-sensitive work | Medium |
| Plugin | Compiled extensions for performance and specialized needs | High |

## Example: a do-file to produce a Table 1 

`src/ex1_table1.do` introduces a common reporting task:

- Load public NHANES II data with `webuse`
- Build descriptive statistics with `dtable`
- Export a first table to HTML
- Refine the output with `collect`
- Export a formatted table to various formats

## Code snippet 1

```stata
*! 1.0.0  18/may/2026

version 18
/*
 * Include a table of descriptive statistics for data from the 
 * Second National Health and Nutrition Examination Survey 
 * (NHANES II) (McDowell et al. 1981). 
 * 
 * Use -dtable- to create the table and export it to many formats. 
 */

use ../data/nhanes2l.dta, clear

/*
 * Format the means and standard deviations to two decimal places, add 
 * a title, and export the final table to an HTML file:
 */
dtable age weight bpsystol i.sex i.race,        ///
    by(diabetes, nototals tests)                ///
    continuous(age, test(none))                 ///
    factor(race, test(none))                    ///
    sample(, statistics(freq) place(seplabels)) /// 
    column(by(hide))                            ///
    sformat("(N=%s)" frequency)                 ///
    nformat(%7.2f mean sd)                      ///
    note(Total sample: N = 10,349)              ///
    title(Table 1. Demographics)                ///
    export(../docs/table1.html, replace)
```

## Dissection of the code

* Date and file version - *! 1.0.0  18/may/2026
* Version control - version 18
* File and folder navigation - ../nhanes2l.dta, ../docs/table1.html 
* Line continuation - ///
* Cmments

### Version control

```stata-output
. do ex1_table1

. *! 1.0.0  18/may/2026
. 
. version 18
this is version 17.0 of Stata; it cannot run version 18.0 programs
     You can purchase the latest version of Stata by visiting https://www.stata.com.
r(9);
```

### Organize folders and files

For file and path names:

* Use forward slash as seperator, backslash fails on MacOS and Linux 
* Try to use only lower case english letters and _ in file names, capital letter likely causes problems on MacOS and Linux 
* Double quote path with spaces

### Double quote or compund double quote path with spaces

```stata-output
. cd "hua peng"
c:\talks\webinar2026\hua peng

. do test_path

. display "Test file path current working directory = |`c(pwd)'|" 
Test file path current working directory = |c:\talks\webinar2026\hua peng|

. do `c(pwd)'/test_path.do
file c:\talks\webinar2026\hua.do not found
r(601);

. do "`c(pwd)'/test_path.do"

. display "Test file path current working directory = |`c(pwd)'|" 
Test file path current working directory = |c:\talks\webinar2026\hua peng|

. do `"`c(pwd)'/test_path.do"'

. display "Test file path current working directory = |`c(pwd)'|" 
Test file path current working directory = |c:\talks\webinar2026\hua peng|
```

### Another common pitfall when tempfile path contains space  

```stata-code
/* temp directory is at c:/Users/Hua Peng/AppData/Local/Temp */
tempfile f
save `f'
/* Fail with a crptic error messge
 * invalid 'Peng' 
 * r(198);
 * It attempted to issue something like:
 * save c:/Users/Hua Peng/AppData/Local/Temp/ST_93bc_000001.tmp
 */

 /* correct way */
 save `"`f'"'
```

## Second Pass: Make It Publishable

Good reporting code separates intent from decoration:

- Declare which variables belong in the table
- Define grouping and tests explicitly
- Format numbers consistently
- Add title, notes, and export target
- Keep styling commands close to the output they affect

## 4. Pass Arguments Deliberately

Reusable Stata code should make inputs obvious:

- Use locals for short-lived values
- Parse command syntax with `syntax`
- Pass variables, options, and file paths explicitly
- Avoid hidden dependencies on global state
- Validate inputs before expensive or destructive work

## Turning the Example Into a Program

A reusable command might expose:

```stata
program define make_table1
    syntax varlist using/, By(varname) [Replace]
    dtable `varlist', by(`by') export(`using', `replace')
end
```

The goal is not clever code. The goal is a safer interface.

## 5. Return Results

Return values let the next command use the previous command's work:

- Use `return` for r-class results
- Use `ereturn` for estimation commands
- Store scalars, macros, and matrices with clear names
- Check outputs with `return list` or `ereturn list`
- Document what callers can rely on

## Extend Existing Programs Safely

When wrapping a built-in command:

- Preserve familiar Stata behavior
- Add validation and safer defaults
- Keep return values compatible with downstream code
- Document new options with runnable examples
- Prefer small extensions over surprising rewrites

## Reproducible Graphics

Graphics should be just as rerunnable as tables:

- Prepare data separately from graph styling
- Use named schemes and consistent export paths
- Save graph code with the analysis that produced it
- Store final exports outside the source script directory
- Keep examples small enough for attendees to rerun

## Live Coding Checkpoints

Use these moments to slow down and let attendees inspect the code:

1. Run the first `dtable`
2. Add formatting and notes
3. Apply `collect` styling
4. Wrap the workflow in a small program
5. Confirm exported files are reproducible

## Attendee Checklist

After the webinar, attendees should be able to:

- Recreate the project structure locally
- Run the Table 1 example end to end
- Identify when a do-file should become an ado command
- Pass inputs without relying on globals
- Inspect returned results before using them downstream

## Next Steps

- Add a public sample dataset or documented `webuse` dependency
- Rename example scripts with session order prefixes
- Add one complete `.ado` example under `src/`
- Add a short setup section to `README.md`
- Export the final presentation format when the agenda is locked